In [1]:
import os
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

# Paths available in this environment (use the simplest stable ones you listed)
TRAIN_PATH = "/kaggle/data/train.csv"
TEST_PATH = "/kaggle/data/test.csv"
SAMPLE_SUB_PATH = "/kaggle/data/sample_submission.csv"

label_cols = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]

print("Loading data...")
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
submission_df = pd.read_csv(SAMPLE_SUB_PATH)

# Ensure text is string and missing handled (minimal, prevents vectorizer errors)
train_text = train_df["comment_text"].fillna("").astype(str)
test_text = test_df["comment_text"].fillna("").astype(str)

print(train_df.shape, test_df.shape, submission_df.shape)
print("Train columns:", list(train_df.columns))



Loading data...
(159571, 8) (153164, 2) (153164, 7)
Train columns: ['id', 'comment_text', 'toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']


In [2]:
# Change rationale: Your current code depends on missing external model submissions and fails to produce a valid CSV.
# This keeps the same "predict probabilities for 6 labels" core evaluation semantics, but generates predictions
# directly from the provided train/test data so you can get a valid Kaggle score.

# TF-IDF settings chosen as a strong, standard baseline for this competition without changing the overall approach.
# (No early stopping, no approximation; just a deterministic feature extractor + linear classifier.)
print("Fitting TF-IDF...")
vectorizer = TfidfVectorizer(
    strip_accents="unicode",
    lowercase=True,
    analyzer="word",
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.9,
    max_features=300000,
    sublinear_tf=True,
)

X_train = vectorizer.fit_transform(train_text)
X_test = vectorizer.transform(test_text)

print("Vectorized shapes:", X_train.shape, X_test.shape)



Fitting TF-IDF...
Vectorized shapes: (159571, 300000) (153164, 300000)


In [3]:
# Change rationale: Train one classifier per label (multi-label setup) to output well-calibrated probabilities
# for ROC-AUC evaluation. This is a minimal, standard approach and should yield a strong non-zero score.

print(
    "Training 6 one-vs-rest LogisticRegression models and predicting probabilities..."
)
test_pred = np.zeros((X_test.shape[0], len(label_cols)), dtype=np.float64)

# Deterministic, strong defaults for this dataset
base_clf = LogisticRegression(
    solver="saga",
    penalty="l2",
    C=4.0,
    max_iter=200,
    n_jobs=-1,
    random_state=42,
)

for j, col in enumerate(label_cols):
    y = train_df[col].values
    clf = LogisticRegression(
        solver=base_clf.solver,
        penalty=base_clf.penalty,
        C=base_clf.C,
        max_iter=base_clf.max_iter,
        n_jobs=base_clf.n_jobs,
        random_state=base_clf.random_state,
    )
    clf.fit(X_train, y)
    test_pred[:, j] = clf.predict_proba(X_test)[:, 1]
    print(f" - done: {col}")



Training 6 one-vs-rest LogisticRegression models and predicting probabilities...
 - done: toxic
 - done: severe_toxic
 - done: obscene
 - done: threat
 - done: insult
 - done: identity_hate


In [4]:
# Change rationale: Ensure submission is valid and correctly aligned.
# We use sample_submission.csv as the template (correct order/columns) and map by id to avoid any ordering issues.

pred_df = pd.DataFrame(test_pred, columns=label_cols)
pred_df.insert(0, "id", test_df["id"].values)

# Align to sample_submission ids (robust even if test order changes)
merged = submission_df[["id"]].merge(pred_df, on="id", how="left")

# Safety: if any ids were missing (shouldn't happen), fill with 0.5 neutral probability
for col in label_cols:
    merged[col] = merged[col].fillna(0.5).clip(0.0, 1.0)

out_path = "submission_final_ensemble.csv"
merged.to_csv(out_path, index=False)

print(f"Wrote {out_path} with shape {merged.shape}")
print(merged.head())

Wrote submission_final_ensemble.csv with shape (153164, 7)
                 id     toxic  severe_toxic   obscene    threat    insult  \
0  00001cee341fdb12  0.999670      0.178601  0.998135  0.049902  0.939335   
1  0000247867823ef7  0.002900      0.000989  0.001288  0.000325  0.003734   
2  00013b17ad220c46  0.013112      0.002034  0.007765  0.000788  0.007196   
3  00017563c3f7919a  0.002180      0.001152  0.001654  0.000763  0.001413   
4  00017695ad8997eb  0.014600      0.003809  0.005275  0.001097  0.005620   

   identity_hate  
0       0.411066  
1       0.001475  
2       0.002079  
3       0.000781  
4       0.001698  
